# MobileNetV3-Small — phân loại biển một/hai hàng (Colab)
Fine-tune ImageNet-1K trên **crop gốc VNLP**, không dùng cache OCR đã ghép hàng.

Notebook độc lập với phiên CRNN. Nguồn sự thật là `metadata_audit.csv` đã được lưu sau khi sửa layout/bbox và `split_manifest.csv` của lần train CRNN. Không suy ra layout từ tên thư mục; không chia lại split; không thêm ảnh đã loại khỏi manifest.

**Đã có classifier train dở?** Giữ đường dẫn output cũ: notebook tự resume `layout_last.pt` nếu fingerprint dữ liệu khớp. Tương thích cấu trúc checkpoint của các cell MobileNet đã gửi trước đó. Nếu fingerprint khác, dừng để kiểm tra, không tự ghi đè.

Tách notebook không mở lại hạn mức GPU Colab. Có thể kiểm tra metadata/checkpoint trên CPU, nhưng mặc định chỉ train khi có GPU.

Tham khảo: [MobileNetV3-Small/Torchvision](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.mobilenet_v3_small.html).


In [ ]:
%pip -q install opencv-python-headless pandas numpy pillow matplotlib tqdm scikit-learn
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path as lyt_Path

# Sửa các đường dẫn nếu khác tài khoản của bạn.
lyt_crnn_run_dir = lyt_Path('/content/drive/MyDrive/Computer Vision/DoAnCuoiKy/VNLP_CRNN_scratch_v1')
lyt_zip_path = lyt_Path('/content/drive/MyDrive/Computer Vision/DoAnCuoiKy/Dataset/VNLP_detection.zip')

# Giữ đúng thư mục classifier cũ nếu muốn tiếp tục lần train dở.
lyt_run_dir = lyt_crnn_run_dir / 'layout_mobilenetv3_small_v1'

# Nếu cùng runtime với CRNN, tái sử dụng ảnh local đã giải nén.
# Runtime mới: chỉ giải nén các ảnh có trong metadata và được đánh dấu valid.
lyt_local_root = lyt_Path('/content/vnlp_crnn_data')

lyt_size = 128
lyt_batch_size = 64
lyt_workers = 2
lyt_epochs = 25
lyt_warmup_epochs = 2
lyt_patience = 6
lyt_seed = 42
lyt_auto_resume = True
lyt_require_gpu = True
lyt_evaluate_only = False
lyt_run_final_test = False
lyt_run_benchmark = False

# None: đọc bbox mode và margin từ config CRNN đã lưu.
# Chỉ chỉnh tay nếu đã kiểm tra tọa độ và biết config nguồn không đúng.
lyt_bbox_mode_override = None
lyt_crop_margin_override = None

# OCR ghép: chỉ bật khi classifier đã chốt bằng validation.
joint_run_test = False
joint_run_benchmark = False
joint_crnn_checkpoint = lyt_crnn_run_dir / 'crnn_best.pt'
joint_test_batch_size = 32
joint_benchmark_samples = 200
joint_benchmark_repeats = 3
joint_target_fps = 30


In [ ]:
import json as lyt_json
import hashlib as lyt_hashlib
import random as lyt_random
import shutil as lyt_shutil
import zipfile as lyt_zipfile
import time as lyt_time

import numpy as lyt_np
import pandas as lyt_pd
import matplotlib.pyplot as lyt_plt
import torch as lyt_torch

from PIL import Image as lyt_Image
from torch import nn as lyt_nn
from torch.utils.data import Dataset as lyt_Dataset
from torch.utils.data import DataLoader as lyt_DataLoader
from torchvision import transforms as lyt_transforms
from torchvision.models import (
    mobilenet_v3_small as lyt_mobilenet,
    MobileNet_V3_Small_Weights as lyt_Weights,
)
from sklearn.metrics import (
    accuracy_score as lyt_accuracy,
    precision_recall_fscore_support as lyt_prfs,
)
from tqdm.auto import tqdm as lyt_tqdm

lyt_device = lyt_torch.device('cuda' if lyt_torch.cuda.is_available() else 'cpu')
lyt_class_names = ['one_row', 'two_rows']
lyt_label_map = {'one_row': 0, 'two_rows': 1}
lyt_best_path = lyt_run_dir / 'layout_best.pt'
lyt_last_path = lyt_run_dir / 'layout_last.pt'
lyt_run_dir.mkdir(parents=True, exist_ok=True)

def lyt_save_json(value, path):
    path.write_text(lyt_json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')

def lyt_seed_all(seed):
    lyt_random.seed(seed)
    lyt_np.random.seed(seed)
    lyt_torch.manual_seed(seed)
    if lyt_torch.cuda.is_available():
        lyt_torch.cuda.manual_seed_all(seed)

print('Device:', lyt_device)
print('Torch:', lyt_torch.__version__)
if lyt_device.type == 'cuda':
    print('GPU:', lyt_torch.cuda.get_device_name(0))
else:
    print('CPU: vẫn kiểm tra metadata/checkpoint được; mặc định chưa cho train.')
print('Checkpoint classifier cũ:', lyt_last_path.exists())


## 1. Nạp metadata đã sửa, giữ nguyên split
Cần hai file nguồn:
- `metadata_audit.csv`: chứa layout/bbox/valid hiệu lực sau sửa.
- `split_manifest.csv`: chứa assignment train/val/test, kể cả lần sửa ảnh trùng.

Cột layout trong split cũ có thể là nhãn thư mục ban đầu: chỉ đối chiếu image_id, text và split; dùng layout từ metadata đã sửa. Các file correction riêng không được tự áp chồng lên metadata để tránh hoàn tác nhầm. Xem bảng và ảnh bên dưới để xác nhận metadata đã lưu đúng.


In [ ]:
def lyt_parse_valid(series):
    values = series.astype(str).str.strip().str.lower()
    mapping = {'true': True, 'false': False, '1': True, '0': False,
               '1.0': True, '0.0': False}
    assert values.isin(mapping).all(), 'Cột valid có giá trị không xác định'
    return values.map(mapping).astype(bool)

lyt_metadata_path = lyt_crnn_run_dir / 'metadata_audit.csv'
lyt_split_path = lyt_crnn_run_dir / 'split_manifest.csv'
lyt_source_config_path = lyt_crnn_run_dir / 'config.json'

assert lyt_metadata_path.is_file(), f'Thiếu metadata đã sửa: {lyt_metadata_path}'
assert lyt_split_path.is_file(), f'Thiếu split đã lưu: {lyt_split_path}'

lyt_df = lyt_pd.read_csv(
    lyt_metadata_path,
    dtype={'image_id': str, 'text': str, 'layout': str, 'split': str},
    keep_default_na=False,
)
lyt_splits = lyt_pd.read_csv(lyt_split_path, dtype=str, keep_default_na=False)
lyt_required = {'image_id','text','layout','split','valid','a','b','c','d'}
assert lyt_required.issubset(lyt_df.columns)
assert lyt_df.image_id.is_unique and lyt_splits.image_id.is_unique
assert set(lyt_df.image_id) == set(lyt_splits.image_id), 'Metadata và split không cùng tập ảnh'
assert lyt_df.layout.isin(lyt_class_names).all()
assert set(lyt_df.split) == {'train','val','test'}

lyt_df['valid'] = lyt_parse_valid(lyt_df.valid)
for lyt_col in ['a','b','c','d']:
    lyt_df[lyt_col] = lyt_pd.to_numeric(lyt_df[lyt_col], errors='raise')
    assert lyt_np.isfinite(lyt_df[lyt_col]).all()
    assert (lyt_df[lyt_col] % 1 == 0).all(), 'Tọa độ phải nguyên'

lyt_check = lyt_df[['image_id','text','split']].merge(
    lyt_splits[['image_id','text','split']], on='image_id',
    suffixes=('', '_saved'), validate='one_to_one',
)
assert (lyt_check.text == lyt_check.text_saved).all(), 'Nhãn khác split nguồn'
assert (lyt_check.split == lyt_check.split_saved).all(), 'Assignment split bị đổi'
assert lyt_df.groupby('text').split.nunique().max() == 1, 'Cùng biển xuyên split'
if 'sha256' in lyt_df and lyt_df.sha256.astype(str).str.len().eq(64).all():
    assert lyt_df.groupby('sha256').split.nunique().max() == 1, 'Ảnh trùng xuyên split'

lyt_source_preprocess = {}
if lyt_source_config_path.is_file():
    lyt_source_preprocess = lyt_json.loads(
        lyt_source_config_path.read_text(encoding='utf-8')
    ).get('preprocess', {})
lyt_bbox_mode = lyt_bbox_mode_override or lyt_source_preprocess.get('bbox_mode')
lyt_crop_margin = (
    lyt_crop_margin_override if lyt_crop_margin_override is not None
    else lyt_source_preprocess.get('margin')
)
assert lyt_bbox_mode in {'xyxy','xywh'}, 'Không biết bbox mode: cần kiểm tra config CRNN'
assert lyt_crop_margin is not None and 0 <= lyt_crop_margin <= .25

lyt_identity_columns = ['image_id','text','layout','split','valid','a','b','c','d']
lyt_data_hash = lyt_hashlib.sha256(
    lyt_df[lyt_identity_columns].sort_values('image_id').to_csv(index=False).encode()
).hexdigest()
# Giữ schema identity tương thích checkpoint MobileNet đã gửi trước.
lyt_identity = {
    'data_hash': lyt_data_hash,
    'bbox_mode': lyt_bbox_mode,
    'crop_margin': lyt_crop_margin,
    'size': lyt_size,
    'classes': lyt_label_map,
    'initial_weights': 'MobileNet_V3_Small_Weights.IMAGENET1K_V1',
}
lyt_resume = lyt_auto_resume and lyt_last_path.is_file()
if lyt_last_path.exists() and not lyt_auto_resume and not lyt_evaluate_only:
    raise RuntimeError('Đã có classifier checkpoint. Bật auto resume hoặc chọn output khác.')
if lyt_resume:
    lyt_previous = lyt_torch.load(lyt_last_path, map_location='cpu', weights_only=False)
    assert lyt_previous['identity'] == lyt_identity, (
        'Metadata/cấu hình không khớp classifier checkpoint. Không chia lại dữ liệu hoặc '
        'bỏ kiểm tra. Xác minh metadata đã lưu; chọn output mới nếu chủ đích đổi dữ liệu.'
    )
    print('Sẽ tiếp tục từ epoch:', lyt_previous['epoch'] + 1)
    print('Val lần lưu cuối:', lyt_previous['val_metrics'])
    del lyt_previous

print('BBox:', lyt_bbox_mode, '| margin:', lyt_crop_margin)
print('Metadata SHA256:', lyt_data_hash)
display(lyt_pd.crosstab(lyt_df.split, lyt_df.layout))
print('Crop invalid:', int((~lyt_df.valid).sum()))
display(lyt_df[~lyt_df.valid][['image_id','layout','split']].head(20))


## 2. Tái sử dụng local nếu còn, nếu thiếu mới giải nén
Không cần giải nén lại file đã có và có dung lượng >0. Runtime mới phải đưa ảnh từ ZIP về local.
Chỉ lấy ảnh valid có trong metadata; ảnh đã bị loại khỏi manifest không được thêm lại.
Không dùng đường dẫn /content cũ lưu trong CSV, vì runtime mới có thể khác.


In [ ]:
lyt_local_root.mkdir(parents=True, exist_ok=True)
def lyt_resolve_local(image_id):
    destination = (lyt_local_root / image_id).resolve()
    assert destination.is_relative_to(lyt_local_root.resolve()), 'image_id vượt thư mục dữ liệu'
    return destination

lyt_df['path'] = lyt_df.image_id.map(lambda x: str(lyt_resolve_local(x)))
lyt_needed = lyt_df[lyt_df.valid].to_dict('records')
lyt_missing = [
    row for row in lyt_needed
    if not lyt_Path(row['path']).is_file() or lyt_Path(row['path']).stat().st_size == 0
]
print('Đã có local:', len(lyt_needed)-len(lyt_missing), '| Cần giải nén:', len(lyt_missing))

if lyt_missing:
    assert lyt_zip_path.is_file(), f'Thiếu ZIP: {lyt_zip_path}'
    with lyt_zipfile.ZipFile(lyt_zip_path) as archive:
        # Chấp nhận ZIP có thêm một thư mục bọc bên ngoài detection/.
        lookup = {}
        for member in archive.infolist():
            name = member.filename.replace('\\', '/')
            marker = name.find('detection/')
            if not member.is_dir() and marker >= 0:
                key = name[marker:]
                lookup.setdefault(key, []).append(member)
        for row in lyt_tqdm(lyt_missing, desc='Extract missing local images'):
            candidates = lookup.get(row['image_id'], [])
            assert len(candidates) == 1, f"ZIP thiếu/trùng entry: {row['image_id']}"
            target = lyt_Path(row['path'])
            target.parent.mkdir(parents=True, exist_ok=True)
            temporary = target.with_name(target.name + '.extracting')
            with archive.open(candidates[0]) as source, temporary.open('wb') as dest:
                lyt_shutil.copyfileobj(source, dest)
            assert temporary.stat().st_size == candidates[0].file_size
            temporary.replace(target)

assert all(lyt_Path(r['path']).is_file() for r in lyt_needed)
lyt_df.to_csv(lyt_run_dir / 'dataset_manifest.csv', index=False)
lyt_save_json(lyt_identity, lyt_run_dir / 'config.json')


In [ ]:
def lyt_read_rgb(path):
    with lyt_Image.open(path) as image:
        return lyt_np.asarray(image.convert('RGB')).copy()

def lyt_crop_plate(rgb, row):
    h,w = rgb.shape[:2]
    a,b,c,d = (int(row[k]) for k in ['a','b','c','d'])
    x1,y1,x2,y2 = (a,b,c,d) if lyt_bbox_mode == 'xyxy' else (a,b,a+c,b+d)
    if not (0 <= x1 < x2 <= w and 0 <= y1 < y2 <= h):
        raise ValueError(f"Bbox sai: {row['image_id']} | {(x1,y1,x2,y2)} / {(w,h)}")
    mx,my = round((x2-x1)*lyt_crop_margin), round((y2-y1)*lyt_crop_margin)
    box = max(0,x1-mx),max(0,y1-my),min(w,x2+mx),min(h,y2+my)
    return rgb[box[1]:box[3],box[0]:box[2]].copy()

def lyt_letterbox(image):
    image = image.convert('RGB')
    w,h = image.size
    scale = min(lyt_size/w, lyt_size/h)
    nw,nh = max(1,round(w*scale)),max(1,round(h*scale))
    image = image.resize((nw,nh),lyt_Image.Resampling.BILINEAR)
    canvas = lyt_Image.new('RGB',(lyt_size,lyt_size),(127,127,127))
    canvas.paste(image,((lyt_size-nw)//2,(lyt_size-nh)//2))
    return canvas

lyt_preview_candidates = lyt_df[(lyt_df.split=='train') & lyt_df.valid]
lyt_preview = lyt_pd.concat([
    g.sample(min(6,len(g)),random_state=lyt_seed)
    for _,g in lyt_preview_candidates.groupby('layout')
])
lyt_fig,lyt_axes = lyt_plt.subplots(
    (len(lyt_preview)+3)//4,4,figsize=(16,3.2*((len(lyt_preview)+3)//4)),squeeze=False
)
for ax,(_,row) in zip(lyt_axes.flat,lyt_preview.iterrows()):
    crop = lyt_crop_plate(lyt_read_rgb(row.path),row)
    ax.imshow(crop)
    ax.set_title(row.layout+' | '+row.text)
    ax.axis('off')
for ax in list(lyt_axes.flat)[len(lyt_preview):]: ax.axis('off')
lyt_plt.tight_layout(); lyt_plt.show()
print('Kiểm tra nhãn trên ảnh là layout đã sửa. Không tách hoặc ghép hai hàng.')


## 3. Dataset
Augmentation nhẹ trên crop gốc trước letterbox; không flip và không tách hàng.
Train loại mẫu valid=False; val/test giữ mẫu invalid, prediction=-1 và tính sai.
Nếu một mẫu valid=True vẫn crop lỗi, dừng với image_id thay vì âm thầm bỏ qua.


In [ ]:
lyt_train_augment = lyt_transforms.Compose([
    lyt_transforms.RandomRotation(5,expand=True,fill=(127,127,127)),
    lyt_transforms.ColorJitter(brightness=.2,contrast=.2,saturation=.1),
    lyt_transforms.RandomApply([
        lyt_transforms.GaussianBlur(3,sigma=(.1,.6))
    ],p=.1),
])
lyt_to_tensor = lyt_transforms.Compose([
    lyt_transforms.ToTensor(),
    lyt_transforms.Normalize(mean=(.485,.456,.406),std=(.229,.224,.225)),
])

class LYTPlateDataset(lyt_Dataset):
    def __init__(self,frame,training=False):
        self.rows=frame.to_dict('records'); self.training=training
    def __len__(self): return len(self.rows)
    def __getitem__(self,index):
        row=self.rows[index]; valid=bool(row['valid'])
        if valid:
            try:
                crop=lyt_crop_plate(lyt_read_rgb(row['path']),row)
                image=lyt_Image.fromarray(crop)
            except Exception as exc:
                raise RuntimeError(f"Classifier crop lỗi: {row['image_id']}") from exc
            if self.training: image=lyt_train_augment(image)
        else:
            image=lyt_Image.new('RGB',(lyt_size,lyt_size),(127,127,127))
        tensor=lyt_to_tensor(lyt_letterbox(image))
        return tensor,lyt_label_map[row['layout']],valid,row['image_id']

def lyt_worker_seed(worker_id):
    seed=lyt_torch.initial_seed()%2**32
    lyt_random.seed(seed); lyt_np.random.seed(seed)

def lyt_make_loader(frame,training=False):
    return lyt_DataLoader(
        LYTPlateDataset(frame,training),batch_size=lyt_batch_size,shuffle=training,
        num_workers=lyt_workers,pin_memory=lyt_device.type=='cuda',
        persistent_workers=lyt_workers>0,worker_init_fn=lyt_worker_seed,
        generator=lyt_torch.Generator().manual_seed(lyt_seed),
    )

lyt_train_df=lyt_df[(lyt_df.split=='train') & lyt_df.valid].copy()
lyt_val_df=lyt_df[lyt_df.split=='val'].copy()
lyt_test_df=lyt_df[lyt_df.split=='test'].copy()
assert set(lyt_train_df.layout)==set(lyt_class_names)
lyt_train_loader=lyt_make_loader(lyt_train_df,True)
lyt_val_loader=lyt_make_loader(lyt_val_df)
lyt_test_loader=lyt_make_loader(lyt_test_df)


## 4. Fine-tune và resume
Hai epoch đầu freeze backbone và BatchNorm của backbone. Sau đó unfreeze, backbone LR=1e-5, head LR=1e-4.
FP32, AdamW, giảm LR theo val macro-F1, early stopping patience=6.
Chọn best theo macro-F1; tie-break accuracy.

Checkpoint last lưu sau mỗi epoch: nếu đứt giữa epoch, phần epoch đó chạy lại.
Resume khôi phục optimizer/scheduler/history; với checkpoint cũ không có RNG, chuỗi augmentation có thể khác một lần chạy liên tục. Không cần CRNN weights.


In [ ]:
def lyt_metrics(details):
    precision,recall,f1,support=lyt_prfs(
        details.target,details.prediction,labels=[0,1],zero_division=0
    )
    metrics=dict(
        accuracy=float(lyt_accuracy(details.target,details.prediction)),
        macro_f1=float(f1.mean()),invalid_inputs=int((~details.valid).sum()),
        samples=int(len(details)),
    )
    table=lyt_pd.DataFrame(dict(
        class_name=lyt_class_names,precision=precision,recall=recall,f1=f1,support=support
    ))
    return metrics,table

lyt_criterion=lyt_nn.CrossEntropyLoss()

def lyt_run_epoch(network,loader,optimizer=None,freeze_backbone=False):
    training=optimizer is not None
    network.train(training)
    if training and freeze_backbone: network.features.eval()
    records=[]; total_loss=0.; valid_count=0
    with lyt_torch.set_grad_enabled(training):
        for images,targets,valid,image_ids in lyt_tqdm(
            loader,desc='layout train' if training else 'layout eval',leave=False
        ):
            images=images.to(lyt_device,non_blocking=True)
            targets=targets.to(lyt_device,non_blocking=True)
            valid=valid.to(lyt_device)
            if training:
                optimizer.zero_grad(set_to_none=True)
                assert bool(valid.all())
            logits=network(images)
            if not bool(lyt_torch.isfinite(logits).all()):
                raise RuntimeError('Classifier logits NaN/Inf')
            loss=None
            if bool(valid.any()):
                loss=lyt_criterion(logits[valid],targets[valid])
                if not bool(lyt_torch.isfinite(loss)): raise RuntimeError('Classifier loss NaN/Inf')
                count=int(valid.sum().item())
                total_loss+=float(loss.detach())*count; valid_count+=count
            if training:
                loss.backward()
                lyt_nn.utils.clip_grad_norm_(network.parameters(),5.,error_if_nonfinite=True)
                optimizer.step()
            confidence,prediction=logits.softmax(1).max(1)
            prediction=prediction.masked_fill(~valid,-1)
            for iid,target,pred,conf,is_valid in zip(
                image_ids,targets.detach().cpu().tolist(),prediction.detach().cpu().tolist(),
                confidence.detach().cpu().tolist(),valid.detach().cpu().tolist()
            ):
                records.append(dict(image_id=iid,target=target,prediction=pred,
                                    confidence=conf if is_valid else None,valid=is_valid))
    details=lyt_pd.DataFrame(records)
    metrics,table=lyt_metrics(details)
    metrics['loss_valid_inputs']=total_loss/max(1,valid_count)
    return metrics,details,table

def lyt_rng_state():
    return dict(python=lyt_random.getstate(),numpy=lyt_np.random.get_state(),
                torch=lyt_torch.get_rng_state(),
                cuda=lyt_torch.cuda.get_rng_state_all() if lyt_torch.cuda.is_available() else None)

def lyt_restore_rng(state):
    lyt_random.setstate(state['python']); lyt_np.random.set_state(state['numpy'])
    lyt_torch.set_rng_state(state['torch'].cpu())
    if lyt_torch.cuda.is_available() and state['cuda'] is not None:
        lyt_torch.cuda.set_rng_state_all([s.cpu() for s in state['cuda']])


In [ ]:
if lyt_require_gpu and lyt_device.type!='cuda' and not lyt_evaluate_only:
    raise RuntimeError(
        'Chưa có GPU. Metadata/checkpoint đã kiểm tra xong. Chờ GPU rồi chạy lại, '
        'hoặc đặt lyt_require_gpu=False nếu chủ đích train CPU (chậm hơn).'
    )

lyt_seed_all(lyt_seed)
lyt_resume=lyt_auto_resume and lyt_last_path.is_file()
lyt_model=lyt_mobilenet(
    weights=None if (lyt_resume or lyt_evaluate_only) else lyt_Weights.IMAGENET1K_V1
)
lyt_model.classifier[-1]=lyt_nn.Linear(lyt_model.classifier[-1].in_features,2)
lyt_model.to(lyt_device).float()

lyt_optimizer=lyt_torch.optim.AdamW([
    {'params':lyt_model.features.parameters(),'lr':1e-5},
    {'params':lyt_model.classifier.parameters(),'lr':1e-3},
],weight_decay=1e-4)
lyt_scheduler=lyt_torch.optim.lr_scheduler.ReduceLROnPlateau(
    lyt_optimizer,mode='max',factor=.5,patience=2,min_lr=1e-7
)
lyt_history=[]; lyt_start_epoch=1; lyt_best_key=(-1.,-1.); lyt_stale=0

if lyt_resume and not lyt_evaluate_only:
    lyt_state=lyt_torch.load(lyt_last_path,map_location='cpu',weights_only=False)
    assert lyt_state['identity']==lyt_identity
    lyt_model.load_state_dict(lyt_state['model'])
    lyt_optimizer.load_state_dict(lyt_state['optimizer'])
    lyt_scheduler.load_state_dict(lyt_state['scheduler'])
    lyt_history=lyt_state['history']; lyt_start_epoch=lyt_state['epoch']+1
    lyt_best_key=tuple(lyt_state['best_key']); lyt_stale=lyt_state['stale']
    if 'rng' in lyt_state: lyt_restore_rng(lyt_state['rng'])
    if 'loader_rng' in lyt_state:
        lyt_train_loader.generator.set_state(lyt_state['loader_rng'].cpu())
    print('Resume epoch:',lyt_start_epoch)

lyt_epoch_range=range(lyt_start_epoch,lyt_epochs+1)
if lyt_evaluate_only:
    lyt_epoch_range=[]
elif lyt_stale>=lyt_patience:
    print('Checkpoint đã early-stop; chuyển sang best evaluation.')
    lyt_epoch_range=[]

for lyt_epoch in lyt_epoch_range:
    lyt_frozen=lyt_epoch<=lyt_warmup_epochs
    lyt_model.features.requires_grad_(not lyt_frozen)
    if lyt_epoch==lyt_warmup_epochs+1:
        lyt_optimizer.param_groups[0]['lr']=1e-5
        lyt_optimizer.param_groups[1]['lr']=1e-4
    lyt_begin=lyt_time.perf_counter()
    lyt_tm,_,_=lyt_run_epoch(lyt_model,lyt_train_loader,lyt_optimizer,lyt_frozen)
    lyt_vm,lyt_vd,lyt_vtable=lyt_run_epoch(lyt_model,lyt_val_loader)
    lyt_scheduler.step(lyt_vm['macro_f1'])
    lyt_key=(lyt_vm['macro_f1'],lyt_vm['accuracy'])
    lyt_improved=lyt_key>lyt_best_key
    if lyt_improved: lyt_best_key=lyt_key; lyt_stale=0
    else: lyt_stale+=1
    lyt_history.append(dict(
        epoch=lyt_epoch,seconds=lyt_time.perf_counter()-lyt_begin,
        backbone_frozen=lyt_frozen,backbone_lr=lyt_optimizer.param_groups[0]['lr'],
        classifier_lr=lyt_optimizer.param_groups[1]['lr'],
        **{'train_'+k:v for k,v in lyt_tm.items()},
        **{'val_'+k:v for k,v in lyt_vm.items()},
    ))
    lyt_state=dict(
        model=lyt_model.state_dict(),optimizer=lyt_optimizer.state_dict(),
        scheduler=lyt_scheduler.state_dict(),epoch=lyt_epoch,history=lyt_history,
        best_key=lyt_best_key,stale=lyt_stale,identity=lyt_identity,val_metrics=lyt_vm,
        rng=lyt_rng_state(),loader_rng=lyt_train_loader.generator.get_state(),
    )
    # Best ghi trước last: nếu ngắt giữa hai lần ghi, có thể chạy lại epoch thay vì mất best.
    if lyt_improved:
        lyt_tmp=lyt_run_dir/'layout_best.tmp'
        lyt_torch.save(lyt_state,lyt_tmp); lyt_tmp.replace(lyt_best_path)
        lyt_vd.to_csv(lyt_run_dir/'best_val_predictions.csv',index=False)
        lyt_vtable.to_csv(lyt_run_dir/'best_val_per_class.csv',index=False)
    lyt_tmp=lyt_run_dir/'layout_last.tmp'
    lyt_torch.save(lyt_state,lyt_tmp); lyt_tmp.replace(lyt_last_path)
    lyt_pd.DataFrame(lyt_history).to_csv(lyt_run_dir/'history.csv',index=False)
    print(
        f"Layout epoch {lyt_epoch:02d} | train={lyt_tm['accuracy']:.4f} | "
        f"val={lyt_vm['accuracy']:.4f} | F1={lyt_vm['macro_f1']:.4f} | "
        f"best F1={lyt_best_key[0]:.4f}"
    )
    if lyt_stale>=lyt_patience:
        print('Early stopping.'); break

assert lyt_best_path.is_file(), 'Chưa có best checkpoint; chưa thể evaluate-only'
lyt_best=lyt_torch.load(lyt_best_path,map_location='cpu',weights_only=False)
assert lyt_best['identity']==lyt_identity
lyt_model.load_state_dict(lyt_best['model'])
lyt_model.eval()
print('Best epoch:',lyt_best['epoch'],'|',lyt_best['val_metrics'])


## 5. Validation và test cuối cùng
Accuracy/macro-F1, precision/recall theo lớp, confusion matrix có cột invalid.
Không loại crop lỗi khỏi mẫu số. Confidence softmax chưa phải xác suất đúng được hiệu chuẩn.
Chỉ bật `lyt_run_final_test=True` sau khi chốt cấu hình. Không tune theo test.


In [ ]:
lyt_vm,lyt_val_details,lyt_val_table=lyt_run_epoch(lyt_model,lyt_val_loader)
print('VALIDATION:',lyt_vm)
display(lyt_val_table)
lyt_val_errors=lyt_val_details[lyt_val_details.target!=lyt_val_details.prediction]
lyt_val_errors.to_csv(lyt_run_dir/'val_errors.csv',index=False)
lyt_error_samples=lyt_val_errors[lyt_val_errors.valid].head(12)
if len(lyt_error_samples):
    lyt_fig,lyt_axes=lyt_plt.subplots(
        (len(lyt_error_samples)+3)//4,4,
        figsize=(16,3*((len(lyt_error_samples)+3)//4)),squeeze=False
    )
    lyt_lookup=lyt_df.set_index('image_id')
    for ax,error in zip(lyt_axes.flat,lyt_error_samples.to_dict('records')):
        row=lyt_lookup.loc[error['image_id']]
        ax.imshow(lyt_crop_plate(lyt_read_rgb(row.path),row))
        ax.set_title(f"GT: {lyt_class_names[error['target']]} / Pred: {lyt_class_names[error['prediction']]}")
        ax.axis('off')
    for ax in list(lyt_axes.flat)[len(lyt_error_samples):]: ax.axis('off')
    lyt_plt.tight_layout(); lyt_plt.show()

if lyt_history:
    lyt_h=lyt_pd.DataFrame(lyt_history)
    fig,axes=lyt_plt.subplots(1,2,figsize=(12,4))
    for metric,ax in zip(['accuracy','macro_f1'],axes):
        for prefix in ['train','val']:
            ax.plot(lyt_h.epoch,lyt_h[prefix+'_'+metric],label=prefix)
        ax.set_title(metric); ax.legend()
    lyt_plt.tight_layout(); lyt_plt.show()


In [ ]:
if lyt_run_final_test:
    lyt_test_metrics,lyt_test_details,lyt_test_table=lyt_run_epoch(lyt_model,lyt_test_loader)
    print('TEST:',lyt_test_metrics)
    display(lyt_test_table)
    lyt_confusion=lyt_pd.crosstab(
        lyt_test_details.target,lyt_test_details.prediction
    ).reindex(index=[0,1],columns=[0,1,-1],fill_value=0)
    lyt_confusion.index=lyt_class_names
    lyt_confusion.columns=['pred_one_row','pred_two_rows','invalid']
    display(lyt_confusion)
    lyt_save_json(lyt_test_metrics,lyt_run_dir/'test_metrics.json')
    lyt_test_details.to_csv(lyt_run_dir/'test_predictions.csv',index=False)
    lyt_test_table.to_csv(lyt_run_dir/'test_per_class.csv',index=False)
    lyt_confusion.to_csv(lyt_run_dir/'test_confusion_matrix.csv')
else:
    print('Chưa test. Đặt lyt_run_final_test=True rồi chạy lại cell này khi đã chốt model.')


In [ ]:
@lyt_torch.inference_mode()
def lyt_predict_layout(crop_rgb):
    image=lyt_Image.fromarray(lyt_np.asarray(crop_rgb,dtype=lyt_np.uint8))
    tensor=lyt_to_tensor(lyt_letterbox(image)).unsqueeze(0).to(lyt_device)
    lyt_model.eval()
    probabilities=lyt_model(tensor).softmax(1)[0]
    confidence,index=probabilities.max(0)
    return dict(layout=lyt_class_names[int(index.item())],confidence=float(confidence.item()))

@lyt_torch.inference_mode()
def lyt_benchmark(iterations=100):
    lyt_model.eval()
    x=lyt_torch.zeros(1,3,lyt_size,lyt_size,device=lyt_device)
    for _ in range(20): lyt_model(x)
    if lyt_device.type=='cuda': lyt_torch.cuda.synchronize()
    begin=lyt_time.perf_counter()
    for _ in range(iterations): lyt_model(x)
    if lyt_device.type=='cuda': lyt_torch.cuda.synchronize()
    seconds=(lyt_time.perf_counter()-begin)/iterations
    return dict(model_ms_batch1=1000*seconds,model_fps_batch1=1/seconds,
                parameters=sum(p.numel() for p in lyt_model.parameters()),
                device=str(lyt_device),precision='FP32')

if lyt_run_benchmark:
    lyt_speed=lyt_benchmark()
    print(lyt_speed); lyt_save_json(lyt_speed,lyt_run_dir/'model_benchmark.json')
else:
    print('Bật lyt_run_benchmark để đo model-only, chưa gồm crop/YOLO/OCR.')

lyt_example=lyt_val_df[lyt_val_df.valid].iloc[0]
lyt_example_crop=lyt_crop_plate(lyt_read_rgb(lyt_example.path),lyt_example)
print('GT:',lyt_example.layout,'|',lyt_predict_layout(lyt_example_crop))


## 6. OCR với layout tham chiếu và layout MobileNet dự đoán
Hai nhánh chạy trên cùng **toàn bộ test**, cùng best CRNN:
- Crop gốc → layout đã sửa trong metadata → tách/ghép tự động → CRNN.
- Crop gốc → best MobileNet → tách/ghép tự động → cùng CRNN.

Không dùng cache OCR hoặc override khe tách theo từng ảnh ở cả hai nhánh. Vì vậy nhánh tham chiếu có thể khác điểm test cũ nếu test cũ có override thủ công. Đây là so sánh ảnh hưởng của layout; chưa bao gồm YOLO detection/crop dự đoán. Layout tham chiếu vẫn phụ thuộc chất lượng nhãn đã kiểm tra.

Bật joint_run_test để đánh giá cuối cùng. Bật joint_run_benchmark để đo pipeline; benchmark dùng crop validation, không cần mở test.
Để chỉ đánh giá sau này: lyt_evaluate_only=True, giữ cờ test classifier False nếu không cần chạy lại test riêng.


In [ ]:
import cv2 as joint_cv2

# Cùng kiến trúc và bộ tách sloped_gap_v2 với notebook CRNN đã cung cấp.
class JointCRNN(lyt_nn.Module):
    def __init__(self,num_classes,hidden=256):
        super().__init__()
        def block(cin,cout):
            return [lyt_nn.Conv2d(cin,cout,3,padding=1,bias=False),
                    lyt_nn.BatchNorm2d(cout),lyt_nn.ReLU(inplace=True)]
        self.cnn=lyt_nn.Sequential(
            *block(3,64), lyt_nn.MaxPool2d(2,2),
            *block(64,128), lyt_nn.MaxPool2d(2,2),
            *block(128,256), *block(256,256), lyt_nn.MaxPool2d((2,1),(2,1)),
            *block(256,512), *block(512,512), lyt_nn.MaxPool2d((2,1),(2,1)),
            lyt_nn.Conv2d(512,512,(2,1),bias=False),lyt_nn.BatchNorm2d(512),lyt_nn.ReLU(inplace=True))
        self.rnn=lyt_nn.LSTM(512,hidden,num_layers=2,batch_first=True,
                         bidirectional=True,dropout=.1)
        self.head=lyt_nn.Linear(hidden*2,num_classes)
        self.apply(self._init)
    @staticmethod
    def _init(m):
        if isinstance(m,lyt_nn.Conv2d): lyt_nn.init.kaiming_normal_(m.weight,nonlinearity='relu')
        elif isinstance(m,lyt_nn.Linear):
            lyt_nn.init.xavier_uniform_(m.weight); lyt_nn.init.zeros_(m.bias)
    def forward(self,x):
        f=self.cnn(x)
        assert f.shape[2]==1, f'Feature height must be 1: {f.shape}'
        seq=f.squeeze(2).transpose(1,2)
        seq,_=self.rnn(seq)
        return self.head(seq).transpose(0,1)  # logits [T,B,C], log_softmax ngoài autocast

def joint_row_separator(crop):
    # Tìm một đường phân cách có thể nghiêng; không giả định khe ngang ở h/2.
    h,w = crop.shape[:2]
    if h < 12 or w < 12:
        return lyt_np.full(w,h/2,dtype=lyt_np.float32), False, 'too_small', lyt_np.zeros(h), lyt_np.zeros((h,w),lyt_np.uint8)
    sh=96; sw=min(256,max(64,round(w*sh/h)))
    gray=joint_cv2.cvtColor(joint_cv2.resize(crop,(sw,sh)),joint_cv2.COLOR_RGB2GRAY)
    gray=joint_cv2.GaussianBlur(gray,(3,3),0)
    mask=joint_cv2.adaptiveThreshold(gray,255,joint_cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                              joint_cv2.THRESH_BINARY_INV,31,7)
    ink=(mask>0).astype(lyt_np.float32)
    # Bỏ hai mép dọc có viền/ốc vít khi chấm điểm; giữ nguyên ảnh RGB khi OCR.
    xs=lyt_np.arange(round(.12*sw),round(.88*sw))
    centers=lyt_np.arange(round(.30*sh),round(.70*sh)+1)
    drifts=lyt_np.linspace(-.35*sh,.35*sh,15)
    cc,dd=lyt_np.meshgrid(centers,drifts,indexing='ij')
    cc=cc.ravel(); dd=dd.ravel()
    ys=cc[:,None]+dd[:,None]*(xs[None,:]/max(1,sw-1)-.5)
    def line_ink(y_values):
        values=[]
        for band in (-1,0,1):
            yy=lyt_np.clip(lyt_np.rint(y_values+band).astype(int),0,sh-1)
            values.append(ink[yy,xs[None,:]].mean(axis=1))
        return lyt_np.mean(values,axis=0)
    gap_ink=line_ink(ys)
    # Nhẹ nhàng ưu tiên vùng giữa và ít nghiêng khi nhiều khe có cùng mức mực.
    scores=gap_ink+.04*lyt_np.abs(cc/sh-.5)+.003*lyt_np.abs(dd)/(.35*sh)
    best=int(lyt_np.argmin(scores))
    # Hai phía của khe phải có nét chữ; độ tin cậy chỉ là heuristic để audit.
    top=max(float(line_ink(ys[best:best+1]-offset*sh)[0]) for offset in (.12,.18,.24))
    bottom=max(float(line_ink(ys[best:best+1]+offset*sh)[0]) for offset in (.12,.18,.24))
    ratio=float(gap_ink[best])/max(1e-6,min(top,bottom))
    ok=min(top,bottom)>.03 and ratio<.65
    cut=(cc[best]+dd[best]*(lyt_np.arange(w)/max(1,w-1)-.5))*(h/sh)
    cut=lyt_np.clip(cut,2,h-2).astype(lyt_np.float32)
    projection=ink.mean(axis=1)
    projection=joint_cv2.resize(projection[:,None],(1,h)).ravel()
    debug_mask=joint_cv2.resize(mask,(w,h),interpolation=joint_cv2.INTER_NEAREST)
    # Không chắc chắn vẫn dùng ứng viên tốt nhất, gắn cờ; không đổi về h//2.
    return cut,bool(ok),('sloped_gap' if ok else 'low_confidence_candidate'),projection,debug_mask

def joint_merge_rows(crop, cut):
    h,w=crop.shape[:2]
    boundary=lyt_np.full(w,float(cut),lyt_np.float32) if lyt_np.ndim(cut)==0 else lyt_np.asarray(cut,lyt_np.float32)
    if boundary.shape!=(w,) or not lyt_np.isfinite(boundary).all() or not ((boundary>1)&(boundary<h-1)).all():
        raise ValueError('Đường phân cách không hợp lệ')
    # Remap từng cột: giữ hai phía đường nghiêng rồi chuẩn hóa chiều cao hàng.
    # Đây không phải phép homography bốn góc; góc nhìn phối cảnh mạnh vẫn cần xử lý riêng.
    xx=lyt_np.tile(lyt_np.arange(w,dtype=lyt_np.float32),(joint_height,1))
    t=lyt_np.linspace(0,1,joint_height,dtype=lyt_np.float32)[:,None]
    upper=t*(boundary[None,:]-1)
    lower=boundary[None,:]+t*(h-1-boundary[None,:])
    lines=[]
    for yy,mean_h in ((upper,boundary.mean()),(lower,h-boundary.mean())):
        strip=joint_cv2.remap(crop,xx,yy.astype(lyt_np.float32),joint_cv2.INTER_LINEAR,borderMode=joint_cv2.BORDER_REPLICATE)
        new_w=max(1,round(w*joint_height/mean_h))
        lines.append(joint_cv2.resize(strip,(new_w,joint_height),interpolation=joint_cv2.INTER_LINEAR))
    gap=lyt_np.full((joint_height,max(2,joint_height//8),3),127,lyt_np.uint8)
    return lyt_np.concatenate([lines[0],gap,lines[1]],axis=1)

def joint_letterbox(line):
    h,w=line.shape[:2]
    scale=min(joint_height/h, joint_width/w)
    nw,nh=max(1,round(w*scale)),max(1,round(h*scale))
    resized=joint_cv2.resize(line,(nw,nh),interpolation=joint_cv2.INTER_AREA if scale<1 else joint_cv2.INTER_LINEAR)
    out=lyt_np.full((joint_height,joint_width,3),127,lyt_np.uint8)
    y=(joint_height-nh)//2
    out[y:y+nh,:nw]=resized
    return out, nw, nh


In [ ]:
joint_output = lyt_run_dir / 'joint_ocr_evaluation'
joint_loaded = False

if joint_run_test or joint_run_benchmark:
    assert joint_crnn_checkpoint.is_file(), f'Thiếu CRNN: {joint_crnn_checkpoint}'
    assert lyt_best_path.is_file(), f'Thiếu MobileNet best: {lyt_best_path}'

    joint_crnn_state = lyt_torch.load(
        joint_crnn_checkpoint, map_location='cpu', weights_only=False
    )
    joint_layout_state = lyt_torch.load(
        lyt_best_path, map_location='cpu', weights_only=False
    )
    assert joint_layout_state['identity'] == lyt_identity
    joint_source_identity = joint_crnn_state['identity']
    assert joint_source_identity['split_hash'] == lyt_hashlib.sha256(
        lyt_split_path.read_bytes()
    ).hexdigest(), 'Split khác lần train CRNN'

    joint_preprocess = joint_source_identity['preprocess']
    assert joint_preprocess['two_rows'] == 'sloped_gap_v2_flagged_candidate', (
        'Checkpoint dùng bộ tách hàng khác. Cần đúng mã tiền xử lý khi train, không đoán thay.'
    )
    assert joint_preprocess['bbox_mode'] == lyt_bbox_mode
    assert joint_preprocess['margin'] == lyt_crop_margin
    assert joint_preprocess['normalize'] == 'RGB [-1,1]'

    # Kiểm tra các artifact sửa dữ liệu đã ghi fingerprint khi train CRNN.
    for key,filename in [
        ('layout_corrections_sha256','layout_corrections.csv'),
        ('bbox_corrections_sha256','bbox_corrections.csv'),
        ('quarantine_sha256','ocr_quarantine.csv'),
    ]:
        if key in joint_preprocess:
            path = lyt_crnn_run_dir / filename
            assert path.is_file(), f'Thiếu artifact của CRNN: {path}'
            assert lyt_hashlib.sha256(path.read_bytes()).hexdigest() == joint_preprocess[key], (
                f'{filename} đã khác phiên bản train CRNN'
            )

    joint_height = int(joint_preprocess['height'])
    joint_width = int(joint_preprocess['width'])
    assert joint_height == 32 and joint_width % 4 == 0, 'Input không khớp kiến trúc CRNN này'
    joint_charset = joint_crnn_state['charset']
    joint_idx_to_char = {i+1:c for i,c in enumerate(joint_charset)}
    assert joint_preprocess['charset'] == joint_charset
    assert all(all(c in joint_charset for c in text) for text in lyt_df.text)

    joint_hidden = int(joint_crnn_state['model']['rnn.weight_hh_l0'].shape[1])
    joint_crnn = JointCRNN(len(joint_charset)+1,hidden=joint_hidden)
    joint_crnn.load_state_dict(joint_crnn_state['model'],strict=True)
    joint_crnn.to(lyt_device).float().eval()

    # Instance riêng, không thay lyt_model hoặc model của CRNN ở notebook cũ.
    joint_layout_model = lyt_mobilenet(weights=None)
    joint_layout_model.classifier[-1] = lyt_nn.Linear(
        joint_layout_model.classifier[-1].in_features,2
    )
    joint_layout_model.load_state_dict(joint_layout_state['model'],strict=True)
    joint_layout_model.to(lyt_device).float().eval()

    joint_provenance = dict(
        crnn_checkpoint=str(joint_crnn_checkpoint),crnn_epoch=joint_crnn_state['epoch'],
        layout_checkpoint=str(lyt_best_path),layout_epoch=joint_layout_state['epoch'],
        split_hash=joint_source_identity['split_hash'],layout_data_hash=lyt_identity['data_hash'],
        preprocessing=joint_preprocess,per_image_split_overrides=False,
        precision='FP32',device=str(lyt_device),
    )
    joint_output.mkdir(parents=True,exist_ok=True)
    lyt_save_json(joint_provenance,joint_output/'provenance.json')
    del joint_crnn_state,joint_layout_state
    joint_loaded = True
    print('Đã nạp best MobileNet + best CRNN:', joint_provenance)
else:
    print('Chưa nạp CRNN. Bật joint_run_test hoặc joint_run_benchmark rồi chạy lại cell này.')


In [ ]:
def joint_layout_input(crop):
    # Cùng letterbox và ImageNet normalization với classifier khi fine-tune.
    return lyt_to_tensor(lyt_letterbox(lyt_Image.fromarray(crop)))

def joint_ocr_input(crop,layout):
    if layout=='two_rows':
        cut,split_ok,method,_,_=joint_row_separator(crop)
        line=joint_merge_rows(crop,cut)
    elif layout=='one_row':
        line=crop; split_ok=True; method='one_row'
    else:
        raise ValueError(f'Layout không hợp lệ: {layout}')
    image,_,_=joint_letterbox(line)
    tensor=lyt_torch.from_numpy(image.copy()).permute(2,0,1).float()/127.5-1
    return tensor,bool(split_ok),method

def joint_decode(logits):
    sequences=logits.argmax(2).transpose(0,1).cpu().tolist()
    predictions=[]
    for sequence in sequences:
        previous=None; chars=[]
        for idx in sequence:
            if idx!=0 and idx!=previous: chars.append(joint_idx_to_char[idx])
            previous=idx
        predictions.append(''.join(chars))
    return predictions

def joint_edit_distance(a,b):
    prev=list(range(len(b)+1))
    for i,ca in enumerate(a,1):
        cur=[i]
        for j,cb in enumerate(b,1):
            cur.append(min(cur[-1]+1,prev[j]+1,prev[j-1]+(ca!=cb)))
        prev=cur
    return prev[-1]

class JointOCRDataset(lyt_Dataset):
    def __init__(self,frame):
        self.rows=frame.to_dict('records')
    def __len__(self): return len(self.rows)
    def __getitem__(self,index):
        row=self.rows[index]
        if not bool(row['valid']):
            return row,None,str(row.get('error','invalid crop'))
        try:
            return row,lyt_crop_plate(lyt_read_rgb(row['path']),row),''
        except Exception as exc:
            return row,None,str(exc)

def joint_collate(batch):
    return batch

@lyt_torch.inference_mode()
def joint_evaluate(frame,batch_size=32):
    assert joint_loaded
    joint_crnn.eval(); joint_layout_model.eval()
    loader=lyt_DataLoader(
        JointOCRDataset(frame),batch_size=batch_size,shuffle=False,
        num_workers=lyt_workers,collate_fn=joint_collate,
    )
    records=[]
    for batch in lyt_tqdm(loader,desc='OCR: reference vs predicted layout'):
        current=[]
        for row,crop,error in batch:
            current.append(dict(
                image_id=row['image_id'],target=row['text'],reference_layout=row['layout'],
                predicted_layout='',layout_confidence=None,layout_correct=False,
                crop_valid=crop is not None,crop_error=error,
                reference_prediction='',predicted_layout_prediction='',
                reference_split_ok=False,predicted_split_ok=False,
                reference_preprocess_error='',predicted_preprocess_error='',
            ))
        valid_indices=[i for i,(_,crop,_) in enumerate(batch) if crop is not None]
        if valid_indices:
            inputs=lyt_torch.stack([joint_layout_input(batch[i][1]) for i in valid_indices]).to(lyt_device)
            logits=joint_layout_model(inputs)
            assert bool(lyt_torch.isfinite(logits).all()), 'Layout logits NaN/Inf'
            conf,cls=logits.softmax(1).max(1)
            for i,index,score in zip(valid_indices,cls.cpu().tolist(),conf.cpu().tolist()):
                current[i]['predicted_layout']=lyt_class_names[index]
                current[i]['layout_confidence']=float(score)
                current[i]['layout_correct']=(
                    current[i]['predicted_layout']==current[i]['reference_layout']
                )
        jobs=[]
        for i in valid_indices:
            for branch,layout in [
                ('reference',current[i]['reference_layout']),
                ('predicted',current[i]['predicted_layout']),
            ]:
                try:
                    tensor,ok,method=joint_ocr_input(batch[i][1],layout)
                    current[i][branch+'_split_ok']=ok
                    current[i][branch+'_split_method']=method
                    jobs.append((i,branch,tensor))
                except Exception as exc:
                    current[i][branch+'_preprocess_error']=str(exc)
        for offset in range(0,len(jobs),batch_size):
            part=jobs[offset:offset+batch_size]
            images=lyt_torch.stack([item[2] for item in part]).to(lyt_device)
            logits=joint_crnn(images)
            assert bool(lyt_torch.isfinite(logits).all()), 'CRNN logits NaN/Inf'
            for (i,branch,_),text in zip(part,joint_decode(logits)):
                column='reference_prediction' if branch=='reference' else 'predicted_layout_prediction'
                current[i][column]=text
        records.extend(current)
    details=lyt_pd.DataFrame(records)
    assert len(details)==len(frame) and details.image_id.is_unique
    details['reference_correct']=details.target==details.reference_prediction
    details['predicted_correct']=details.target==details.predicted_layout_prediction
    return details

def joint_ocr_metrics(frame,prediction_column):
    assert len(frame)
    edits=sum(joint_edit_distance(t,p) for t,p in zip(frame.target,frame[prediction_column]))
    return dict(
        samples=int(len(frame)),
        accuracy=float((frame.target==frame[prediction_column]).mean()),
        cer=float(edits/max(1,int(frame.target.str.len().sum()))),
        empty_rate=float((frame[prediction_column]=='').mean()),
    )

def joint_comparison(details):
    reference=joint_ocr_metrics(details,'reference_prediction')
    predicted=joint_ocr_metrics(details,'predicted_layout_prediction')
    transitions=dict(
        both_correct=int((details.reference_correct & details.predicted_correct).sum()),
        reference_correct_predicted_wrong=int((details.reference_correct & ~details.predicted_correct).sum()),
        reference_wrong_predicted_correct=int((~details.reference_correct & details.predicted_correct).sum()),
        both_wrong=int((~details.reference_correct & ~details.predicted_correct).sum()),
    )
    return dict(
        reference_layout=reference,predicted_layout=predicted,
        accuracy_drop_percentage_points=float((reference['accuracy']-predicted['accuracy'])*100),
        cer_increase_percentage_points=float((predicted['cer']-reference['cer'])*100),
        layout_accuracy_all_test=float(details.layout_correct.mean()),
        invalid_crop_count=int((~details.crop_valid).sum()),transitions=transitions,
    )


In [ ]:
if joint_run_test:
    assert joint_loaded
    joint_test_details=joint_evaluate(lyt_test_df,joint_test_batch_size)
    joint_report=joint_comparison(joint_test_details)

    joint_summary=lyt_pd.DataFrame([
        dict(pipeline='Reference layout → CRNN',**joint_report['reference_layout']),
        dict(pipeline='MobileNet → CRNN',**joint_report['predicted_layout']),
    ])
    print('Accuracy/CER trong bảng là tỷ lệ 0–1:')
    display(joint_summary)
    print(f"Accuracy giảm: {joint_report['accuracy_drop_percentage_points']:.3f} điểm phần trăm")
    print(f"CER tăng: {joint_report['cer_increase_percentage_points']:.3f} điểm phần trăm")

    joint_group_rows=[]
    for name,group in joint_test_details.groupby('reference_layout'):
        report=joint_comparison(group)
        joint_group_rows.append(dict(
            layout=name,samples=len(group),
            reference_accuracy=report['reference_layout']['accuracy'],
            predicted_layout_accuracy=report['predicted_layout']['accuracy'],
            accuracy_drop_pp=report['accuracy_drop_percentage_points'],
            reference_cer=report['reference_layout']['cer'],
            predicted_layout_cer=report['predicted_layout']['cer'],
            layout_accuracy=report['layout_accuracy_all_test'],
        ))
    joint_by_layout=lyt_pd.DataFrame(joint_group_rows)
    display(joint_by_layout)
    display(lyt_pd.DataFrame(joint_report['transitions'].items(),columns=['case','samples']))

    joint_confusion=lyt_pd.crosstab(
        joint_test_details.reference_layout,
        joint_test_details.predicted_layout.replace('','invalid'),
    ).reindex(index=lyt_class_names,columns=lyt_class_names+['invalid'],fill_value=0)
    display(joint_confusion)
    joint_harmful=joint_test_details[
        joint_test_details.crop_valid & ~joint_test_details.layout_correct
        & joint_test_details.reference_correct & ~joint_test_details.predicted_correct
    ]
    print('Sai layout khiến OCR từ đúng thành sai:',len(joint_harmful))
    joint_test_details.to_csv(joint_output/'paired_test_predictions.csv',index=False)
    joint_summary.to_csv(joint_output/'comparison_summary.csv',index=False)
    joint_by_layout.to_csv(joint_output/'comparison_by_layout.csv',index=False)
    joint_confusion.to_csv(joint_output/'layout_confusion.csv')
    joint_harmful.to_csv(joint_output/'harmful_layout_errors.csv',index=False)
    lyt_save_json(joint_report,joint_output/'comparison_metrics.json')
else:
    print('Bật joint_run_test rồi chạy lại cell nạp model và cell này khi đã chốt classifier.')


## 7. Tốc độ MobileNet → xử lý hàng → CRNN → CTC decode
Batch=1, FP32, crop validation nạp sẵn trong RAM. Đo cả chuẩn hóa ảnh và CPU/GPU transfers.
Warmup toàn pipeline; đồng bộ GPU để không bỏ sót thời gian thực thi bất đồng bộ.

**Không tính** đọc file/Drive, YOLO, tạo bbox crop, camera, tracking hoặc hiển thị. crops/giây không phải FPS video.
P95 là mốc mà khoảng 95% lượt đo không vượt quá. Bảng ngân sách nhiều biển chỉ là ước lượng xử lý tuần tự, không phải số đo end-to-end.
Chạy khi không có vòng train hoặc benchmark khác chạy song song. GPU/CPU phải được ghi rõ.


In [ ]:
def joint_sync():
    if lyt_device.type=='cuda': lyt_torch.cuda.synchronize(lyt_device)

@lyt_torch.inference_mode()
def joint_predict_crop(crop):
    assert joint_loaded
    joint_layout_model.eval(); joint_crnn.eval()
    joint_sync()
    t0=lyt_time.perf_counter()
    layout_input=joint_layout_input(crop).unsqueeze(0).to(lyt_device)
    cls=int(joint_layout_model(layout_input).argmax(1).item())
    layout=lyt_class_names[cls]
    t1=lyt_time.perf_counter()
    ocr_input,ok,method=joint_ocr_input(crop,layout)
    ocr_input=ocr_input.unsqueeze(0).to(lyt_device)
    t2=lyt_time.perf_counter()
    text=joint_decode(joint_crnn(ocr_input))[0]
    joint_sync()
    t3=lyt_time.perf_counter()
    return dict(
        text=text,predicted_layout=layout,split_ok=ok,split_method=method,
        layout_ms=(t1-t0)*1000,preprocess_ms=(t2-t1)*1000,
        crnn_decode_ms=(t3-t2)*1000,total_ms=(t3-t0)*1000,
    )

def joint_speed_summary(frame,group):
    mean_ms=float(frame.total_ms.mean())
    return dict(
        group=group,runs=int(len(frame)),layout_mean_ms=float(frame.layout_ms.mean()),
        preprocess_mean_ms=float(frame.preprocess_ms.mean()),
        crnn_decode_mean_ms=float(frame.crnn_decode_ms.mean()),
        total_mean_ms=mean_ms,total_median_ms=float(frame.total_ms.median()),
        total_p95_ms=float(lyt_np.percentile(frame.total_ms,95)),
        crops_per_second=1000/mean_ms,
    )

if joint_run_benchmark:
    assert joint_loaded
    assert joint_benchmark_samples>0 and joint_benchmark_repeats>0 and joint_target_fps>0
    joint_candidates=lyt_val_df[lyt_val_df.valid].sample(
        min(joint_benchmark_samples,int(lyt_val_df.valid.sum())),random_state=lyt_seed
    )
    joint_speed_samples=[]; joint_speed_load_errors=[]
    for row in lyt_tqdm(joint_candidates.to_dict('records'),desc='Nạp crop benchmark vào RAM'):
        try:
            crop=lyt_crop_plate(lyt_read_rgb(row['path']),row)
            joint_speed_samples.append(dict(image_id=row['image_id'],crop=crop,reference_layout=row['layout']))
        except Exception as exc:
            joint_speed_load_errors.append(dict(image_id=row['image_id'],error=str(exc)))
    assert joint_speed_samples
    print('Crop nạp được:',len(joint_speed_samples),'| Lỗi nạp:',len(joint_speed_load_errors))
    for i in lyt_tqdm(range(20),desc='Warmup pipeline'):
        joint_predict_crop(joint_speed_samples[i%len(joint_speed_samples)]['crop'])
    joint_speed_records=[]
    for repeat in range(joint_benchmark_repeats):
        for sample in lyt_tqdm(joint_speed_samples,desc=f'Benchmark {repeat+1}/{joint_benchmark_repeats}'):
            # Nếu pipeline lỗi, dừng với ID; không âm thầm bỏ ca lỗi để báo tốc độ đẹp.
            try:
                result=joint_predict_crop(sample['crop'])
            except Exception as exc:
                raise RuntimeError(f"Pipeline benchmark lỗi: {sample['image_id']}") from exc
            joint_speed_records.append(dict(
                repeat=repeat+1,image_id=sample['image_id'],reference_layout=sample['reference_layout'],**result
            ))
    joint_speed_details=lyt_pd.DataFrame(joint_speed_records)
    joint_speed_rows=[joint_speed_summary(joint_speed_details,'ALL')]
    for name,group in joint_speed_details.groupby('predicted_layout'):
        joint_speed_rows.append(joint_speed_summary(group,'route_'+name))
    joint_speed_table=lyt_pd.DataFrame(joint_speed_rows)
    display(joint_speed_table.round(3))
    joint_mean_ms=float(joint_speed_details.total_ms.mean())
    joint_budget_ms=1000/joint_target_fps
    joint_budget_table=lyt_pd.DataFrame([
        dict(plates_per_frame=n,estimated_ocr_mean_ms=n*joint_mean_ms,
             remaining_ms_for_YOLO_and_others=joint_budget_ms-n*joint_mean_ms)
        for n in [1,2,3,5]
    ])
    print(f'Mục tiêu {joint_target_fps} FPS: {joint_budget_ms:.2f} ms/khung hình')
    display(joint_budget_table.round(3))
    print('Ngân sách dương chưa chứng minh realtime: cần đo thêm YOLO/video trên máy triển khai.')

    joint_speed_dir=joint_output/'pipeline_speed'
    joint_speed_dir.mkdir(parents=True,exist_ok=True)
    joint_speed_details.to_csv(joint_speed_dir/'latency_per_crop.csv',index=False)
    joint_speed_table.to_csv(joint_speed_dir/'latency_summary.csv',index=False)
    joint_budget_table.to_csv(joint_speed_dir/'estimated_frame_budget.csv',index=False)
    lyt_pd.DataFrame(joint_speed_load_errors,columns=['image_id','error']).to_csv(
        joint_speed_dir/'load_errors.csv',index=False
    )
    lyt_save_json(dict(
        device=str(lyt_device),
        gpu=lyt_torch.cuda.get_device_name(0) if lyt_device.type=='cuda' else None,
        torch=str(lyt_torch.__version__),precision='FP32',batch_size=1,
        unique_crops=len(joint_speed_samples),repeats=joint_benchmark_repeats,
        sampling_split='val',target_fps=joint_target_fps,
        includes=['MobileNet preprocessing/inference','row processing','CRNN','CTC decode','CPU/GPU transfers'],
        excludes=['file reading','YOLO and bbox crop','camera/video','tracking','display'],
    ),joint_speed_dir/'environment.json')
else:
    print('Bật joint_run_benchmark rồi chạy lại cell nạp model và cell này để đo toàn cụm layout + OCR.')


## Khi Colab ngắt
- Drive giữ best/last và snapshot; /content có thể mất.
- Mở notebook này, mount Drive, giữ đúng ba đường dẫn đầu. Chạy từ đầu để nạp metadata và đưa các ảnh thiếu về local.
- Không chạy lại notebook CRNN, không tái tạo layout từ thư mục. Notebook tự resume nếu checkpoint và metadata khớp.
- Nếu chỉ đánh giá: đặt lyt_evaluate_only=True. Có thể dùng CPU, nhưng tốc độ CPU không đại diện GPU realtime.
- Nếu không khớp fingerprint: kiểm tra đúng metadata đã lưu sau sửa, không bỏ assert.
- Không thay đổi dữ liệu nguồn sau khi đã bắt đầu classifier train; output mới cho phiên bản dữ liệu mới.

Checkpoint giữ schema model/identity/classes tương thích các cell ghép MobileNet + CRNN đã gửi.
Phần bên dưới đã có CRNN và các hàm tiền xử lý OCR. Không cần mở notebook CRNN khi đánh giá ghép; chỉ cần checkpoint và metadata đúng phiên bản.
